In [8]:
import genbrain_model_3dot as model
from genbrain_3dot_smc import collect_frames
from genbrain_smcnn_core.interpreter import run_smcnn_particle_filter
from genbrain_3dot_smc.viz import *
import genbrain_utils_genjax as gjutils
import jax.numpy as jnp
import jax

In [2]:
# First we will collect the generative functions from the 3dot model. 
genfns = [
    model.initial_proposal,
    model.initial_model,
    model.step_proposal,
    model.step_model,
    model.obs_model,
]

In [4]:
# Here we convert digital x,y numpy frames into a spherical occupancy map (i.e. a visual angle occupancy grid)
xy_obs_frames = collect_frames()
vis_angle_observations = jax.vmap(lambda obs: model.find_occupied_2d_angles(obs))(
    jnp.array(xy_obs_frames)
)
len_sim = 5
obs_traces = model.generate_obs_traces(vis_angle_observations[0:len_sim])

In [5]:
# choose a random seed so you can repeat the experiment
key = jax.random.PRNGKey(100)
num_particles = 20

init_states_and_scores, first_step_states_and_scores, unrolled_pf = (
    gjutils.smc.run_particle_filter(
        obs_traces,
        num_particles,
        len_sim,
        genfns,
        key,
        model.translate_proposal_cm_to_model,
    )
)
# keep print of scores or not? its useful for debugging.

Values: P = [-inf -inf -inf -inf -inf -inf -inf -inf -inf -inf -inf -inf -inf -inf
 -inf -inf -inf -inf -inf -inf], Q = [ -8.614116 -12.240436  -8.066771  -7.606756 -10.532404 -10.07239
 -14.469076 -10.72326  -10.532404  -8.154101 -14.343352  -8.614116
  -8.154101 -10.263247 -10.532404 -10.72326   -7.606756 -10.72326
 -10.072391  -7.606756], O = [-31.190197 -34.666298 -34.666298 -34.666298 -38.142395 -38.142395
 -41.618492 -34.666298 -38.142395 -31.190197 -38.142395 -31.190197
 -31.190197 -34.666298 -38.142395 -34.666298 -34.666298 -34.666298
 -38.142395 -34.666298]
Values: P = [      -inf       -inf       -inf       -inf       -inf       -inf
       -inf -15.820416       -inf       -inf -31.632635       -inf
       -inf       -inf       -inf       -inf       -inf       -inf
       -inf       -inf], Q = [-3.0486007 -5.619448  -5.639096  -5.3669643 -4.9069514 -9.021802
 -5.6194477 -6.0991087 -3.754869  -3.0486007 -5.3928404 -5.140358
 -3.5086145 -3.5086145 -3.754869  -7.0177836 -5.86570

Values: P = [      -inf       -inf       -inf -14.069142       -inf       -inf
 -31.47956  -32.92604        -inf -18.1999   -15.98837  -13.67204
 -14.069142 -20.997154 -12.622659 -24.054615       -inf       -inf
       -inf       -inf], Q = [ -3.754869   -3.754869   -9.319347   -5.619448   -7.2321143 -10.469468
  -9.362024   -7.25119    -3.6446338  -5.6194477  -5.159434   -5.6194477
  -5.619448   -7.711205   -5.6194477  -7.0030193  -4.8921866  -3.508615
  -3.508615   -5.140358 ], O = [-34.666298 -34.666298 -38.142395 -38.142395 -38.142395 -31.190197
 -48.57069  -41.618492 -38.142395 -38.142395 -38.142395 -38.142395
 -38.142395 -41.618492 -38.142395 -38.142395 -31.190197 -31.190197
 -31.190197 -34.666298]
Values: P = [-15.793759 -18.565975       -inf       -inf       -inf       -inf
       -inf       -inf       -inf -12.996509       -inf -22.79751
       -inf       -inf -12.996509       -inf -12.025179 -21.972832
       -inf       -inf], Q = [ -7.00302    -5.366965   -5.0282054  -5.6194

In [6]:
latent_variables = [
    {
        "variable": "v3d",
        "q_id": ("dot", "v3d"),
        "q_parents": [],
        "p_parents": [],
        "support": model.xyz_vels,
        "type": "distribution",
    },
    {
        "variable": "xyz",
        "q_id": ("dot", "xyz"),
        "q_parents": [("ego_pos", "ego_matter")],
        "p_parents": [],
        "support": model.xyz_point_cloud,
        "type": "distribution",
    },
    {
        "variable": ("ego_pos", "ego_matter"),
        "q_id": ("dot", "ego_pos", "ego_matter"),
        "q_parents": [],
        "p_parents": ["xyz"],
        "support": model.bool_support,
        "type": "probmap",
    },
    {
        "variable": "lights",
        "q_id": ("dot", "lights"),
        "p_parents": [],
        "q_parents": [],
        "support": model.bool_support,
        "type": "distribution",
    },
    {
        "variable": "diam",
        "q_id": ("dot", "diam"),
        "p_parents": [],
        "q_parents": [],
        "support": model.diams,
        "type": "distribution",
    },
]

obs_variables = [
    {
        "variable": ("obs", "pix"),
        "parents": [],
        "support": model.bool_support,
        "type": "probmap",
    }
]




In [7]:
results = run_smcnn_particle_filter(
    (latent_variables, obs_variables),
    model.initial_model,
    model.step_model,
    model.initial_proposal,
    model.step_proposal,
    model.obs_model,
    5,
    2,
    vis_angle_observations[1:3],
)


Jitting Generative Functions
Initializing SMCNN Particle Filter
q not complete after max recursion
q not complete after max recursion
scoring pixels
0
()
[0 0 0 ... 0 0 0]
(1024,)
scoring pixels
0
()
[0 0 0 ... 0 0 0]
(1024,)
Initialized SMCNN Particle Filter
resampler score
[0.5 0.5]
step 0
scoring pixels
0
()
[0 0 0 ... 0 0 0]
(1024,)
scoring pixels
0
()
[0 0 0 ... 0 0 0]
(1024,)
resampler score
[0.5 0.5]


In [ ]:
xyz_spikes = gather_spikes_from_single_particle_samplescore(results, 0, "xyz", [0, 5])

({0: (array([], dtype=float64), 'assemblies_p1209'),
  1: (array([], dtype=float64), 'assemblies_p1209'),
  2: (array([], dtype=float64), 'assemblies_p1209'),
  3: (array([], dtype=float64), 'assemblies_p1209'),
  4: (array([], dtype=float64), 'assemblies_p1209'),
  5: (array([], dtype=float64), 'assemblies_p1208'),
  6: (array([], dtype=float64), 'assemblies_p1208'),
  7: (array([], dtype=float64), 'assemblies_p1208'),
  8: (array([], dtype=float64), 'assemblies_p1208'),
  9: (array([], dtype=float64), 'assemblies_p1208'),
  10: (array([], dtype=float64), 'assemblies_p1207'),
  11: (array([], dtype=float64), 'assemblies_p1207'),
  12: (array([], dtype=float64), 'assemblies_p1207'),
  13: (array([], dtype=float64), 'assemblies_p1207'),
  14: (array([], dtype=float64), 'assemblies_p1207'),
  15: (array([], dtype=float64), 'assemblies_p1206'),
  16: (array([], dtype=float64), 'assemblies_p1206'),
  17: (array([], dtype=float64), 'assemblies_p1206'),
  18: (array([], dtype=float64), 'asse